# 📘 Notebook 14: Building the Warehouse

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 2 of 6 in this module · (14 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Turn profiled CSV files into a **SQLite** warehouse with **primary keys**, **foreign keys** and a stated grain
- Load the tables in the order that the constraints require, and prove that every key holds
- Recognise a **snowflaked** dimension and a **role-playing** dimension, and say what each costs
- Hide the joins behind a **view** and answer analytical questions by rolling up, drilling down and slicing
- Read a **query plan**, and measure what an **index** and a **summary table** buy

> **Prerequisites:** Notebook 13 (the profile of this warehouse). Notebooks 01 to 04 (SQL, joins, indexes, the star schema).
>
> 🔭 **Why this notebook matters:** in Notebook 04 we built a star schema from a bookshop we had invented. Here the design is given to us by a real company, and our job is the one a data engineer does in the first week of a project: load it into a database we control, make the database enforce the rules we checked by hand in Notebook 13, and make it pleasant and fast for analysts. Everything in this notebook happens in SQLite, a database in a single file. Cloud warehouses such as Amazon Redshift, Azure Synapse or Google BigQuery accept almost the same `CREATE TABLE` statements and the same queries, so what you build here is a faithful small copy of the real thing.

> 📦 **Libraries used in this notebook.** None that need installing: **SQLite** (built into Python, Notebook 01) and **pandas**.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

In [ ]:
import sqlite3
import time

## 1. Choosing what goes in

### Intuition first
A warehouse is not a dump of every column that exists. Each column costs storage, makes every query that reads it a little slower, and, if it is personal data, carries a legal duty. So we choose: the measures and keys of the fact table, and the descriptive columns that an analyst would group or filter by. As in Notebook 13, names, addresses and contact details stay out, and so do the Spanish and French translations.

In [ ]:
sales = load("FactInternetSales")
fact = sales[["SalesOrderNumber", "SalesOrderLineNumber", "ProductKey", "CustomerKey", "SalesTerritoryKey",
              "PromotionKey", "OrderDateKey", "ShipDateKey", "DueDateKey",
              "OrderQuantity", "UnitPrice", "TotalProductCost", "SalesAmount", "TaxAmt", "Freight"]]

dim_date = load("DimDate")[["DateKey", "FullDateAlternateKey", "EnglishDayNameOfWeek", "EnglishMonthName",
                            "MonthNumberOfYear", "CalendarQuarter", "CalendarYear"]]
dim_customer = load("DimCustomer")[["CustomerKey", "GeographyKey", "BirthDate", "MaritalStatus", "Gender",
                                    "YearlyIncome", "TotalChildren", "EnglishEducation", "EnglishOccupation",
                                    "HouseOwnerFlag", "NumberCarsOwned", "CommuteDistance"]]
dim_geography = load("DimGeography")[["GeographyKey", "City", "StateProvinceName", "EnglishCountryRegionName"]]
dim_territory = load("DimSalesTerritory")[["SalesTerritoryKey", "SalesTerritoryRegion", "SalesTerritoryCountry",
                                           "SalesTerritoryGroup"]]
dim_promotion = load("DimPromotion")[["PromotionKey", "EnglishPromotionName", "DiscountPct", "EnglishPromotionType"]]
dim_product = load("DimProduct")[["ProductKey", "ProductSubcategoryKey", "EnglishProductName", "Color",
                                  "StandardCost", "ListPrice", "ProductLine", "ModelName"]]
dim_subcategory = load("DimProductSubcategory")[["ProductSubcategoryKey", "EnglishProductSubcategoryName",
                                                 "ProductCategoryKey"]]
dim_category = load("DimProductCategory")[["ProductCategoryKey", "EnglishProductCategoryName"]]

for name, df in [("fact", fact), ("date", dim_date), ("customer", dim_customer), ("product", dim_product)]:
    print(f"{name:<10} {df.shape[0]:>6} rows x {df.shape[1]:>2} columns")

`ProductSubcategoryKey` arrives as a decimal number (`1.0`, `2.0` ...) because a third of its values are missing (Notebook 13) and pandas stores missing numbers as floats. A key should be an integer. pandas has a nullable integer type for exactly this case:

In [ ]:
dim_product["ProductSubcategoryKey"] = dim_product["ProductSubcategoryKey"].astype("Int64")
dim_product["ProductSubcategoryKey"].head()

## 2. Tables with rules

### Intuition first
In Notebook 13 we *checked* that every key had a partner. A database can do better: it can *refuse* any row that would break the rule, so the check never has to be repeated. That is what a **foreign key** constraint does. A **primary key** does the same for uniqueness: it refuses a second row with the same key, which keeps the grain honest.

We write the `CREATE TABLE` statements ourselves rather than letting pandas invent them, because pandas creates tables without any keys.

In [ ]:
if os.path.exists("adventureworks.db"):
    os.remove("adventureworks.db")
dw = sqlite3.connect("adventureworks.db")
dw.execute("PRAGMA foreign_keys = ON")          # SQLite checks foreign keys only when asked to

dw.executescript('''
CREATE TABLE dim_date (
    date_key INTEGER PRIMARY KEY, full_date TEXT, day_name TEXT, month_name TEXT,
    month INTEGER, quarter INTEGER, year INTEGER);
CREATE TABLE dim_territory (
    territory_key INTEGER PRIMARY KEY, region TEXT, country TEXT, territory_group TEXT);
CREATE TABLE dim_geography (
    geography_key INTEGER PRIMARY KEY, city TEXT, state TEXT, country TEXT);
CREATE TABLE dim_customer (
    customer_key INTEGER PRIMARY KEY,
    geography_key INTEGER REFERENCES dim_geography(geography_key),
    birth_date TEXT, marital_status TEXT, gender TEXT, yearly_income REAL, total_children INTEGER,
    education TEXT, occupation TEXT, house_owner INTEGER, cars_owned INTEGER, commute_distance TEXT);
CREATE TABLE dim_category (
    category_key INTEGER PRIMARY KEY, category TEXT);
CREATE TABLE dim_subcategory (
    subcategory_key INTEGER PRIMARY KEY, subcategory TEXT,
    category_key INTEGER REFERENCES dim_category(category_key));
CREATE TABLE dim_product (
    product_key INTEGER PRIMARY KEY,
    subcategory_key INTEGER REFERENCES dim_subcategory(subcategory_key),
    product TEXT, color TEXT, standard_cost REAL, list_price REAL, product_line TEXT, model TEXT);
CREATE TABLE dim_promotion (
    promotion_key INTEGER PRIMARY KEY, promotion TEXT, discount_pct REAL, promotion_type TEXT);
CREATE TABLE fact_sales (
    order_number TEXT, line_number INTEGER,
    product_key   INTEGER NOT NULL REFERENCES dim_product(product_key),
    customer_key  INTEGER NOT NULL REFERENCES dim_customer(customer_key),
    territory_key INTEGER NOT NULL REFERENCES dim_territory(territory_key),
    promotion_key INTEGER NOT NULL REFERENCES dim_promotion(promotion_key),
    order_date_key INTEGER NOT NULL REFERENCES dim_date(date_key),
    ship_date_key  INTEGER NOT NULL REFERENCES dim_date(date_key),
    due_date_key   INTEGER NOT NULL REFERENCES dim_date(date_key),
    quantity INTEGER, unit_price REAL, cost REAL, revenue REAL, tax REAL, freight REAL,
    PRIMARY KEY (order_number, line_number));
''')
print([r[0] for r in dw.execute("SELECT name FROM sqlite_master WHERE type = 'table'")])

Look at `fact_sales`. Its primary key is the pair that Notebook 13 proved unique: the grain is now written into the table. And three of its columns point to the **same** table, `dim_date`: the order date, the ship date and the due date. One date dimension playing three roles is the **role-playing dimension** we met in Notebook 13.

## 3. Loading in the right order

A foreign key can only point to a row that already exists. So the tables must be loaded from the outside in: first the dimensions that point to nothing, then those that point to them, and the fact table last. We rename the columns to match the new tables as we go.

In [ ]:
def put(table, df, columns):
    df.set_axis(columns, axis=1).to_sql(table, dw, if_exists="append", index=False)
    print(f"{table:<16} {len(df):>6} rows")

put("dim_date", dim_date, ["date_key", "full_date", "day_name", "month_name", "month", "quarter", "year"])
put("dim_territory", dim_territory, ["territory_key", "region", "country", "territory_group"])
put("dim_geography", dim_geography, ["geography_key", "city", "state", "country"])
put("dim_customer", dim_customer, ["customer_key", "geography_key", "birth_date", "marital_status", "gender",
                                   "yearly_income", "total_children", "education", "occupation", "house_owner",
                                   "cars_owned", "commute_distance"])
put("dim_category", dim_category, ["category_key", "category"])
put("dim_subcategory", dim_subcategory, ["subcategory_key", "subcategory", "category_key"])
put("dim_product", dim_product, ["product_key", "subcategory_key", "product", "color", "standard_cost",
                                 "list_price", "product_line", "model"])
put("dim_promotion", dim_promotion, ["promotion_key", "promotion", "discount_pct", "promotion_type"])
put("fact_sales", fact, ["order_number", "line_number", "product_key", "customer_key", "territory_key",
                         "promotion_key", "order_date_key", "ship_date_key", "due_date_key", "quantity",
                         "unit_price", "cost", "revenue", "tax", "freight"])
dw.commit()

Every row was accepted, so every key holds. SQLite can also confirm it in one statement: an empty answer means no row breaks any foreign key.

In [ ]:
dw.execute("PRAGMA foreign_key_check").fetchall()

Now see the rules at work. Try to record a sale of a product that does not exist:

In [ ]:
try:
    dw.execute('''INSERT INTO fact_sales VALUES ('SO99999', 1, 99999, 11000, 1, 1, 20130101, 20130105, 20130110,
                                                 1, 10.0, 5.0, 10.0, 0.8, 0.3)''')
except sqlite3.IntegrityError as error:
    print("Refused:", error)

The bad row never enters the warehouse. That is the difference between checking quality and **enforcing** it.

> ⚠️ Many cloud warehouses (Redshift, BigQuery, Synapse) let you *declare* primary and foreign keys but do **not enforce** them: they use them only as hints for the query optimiser, because checking every key on billions of rows is expensive. There, the checks of Notebook 13 remain your job, usually as a step of the loading pipeline.

## 4. Star or snowflake?

### Intuition first
In a pure **star**, every dimension is one table joined directly to the facts. Our product dimension is not: a product points to a subcategory, which points to a category. A dimension split into a chain of tables is called a **snowflake**, after the shape of the diagram. Snowflaking stores each category name once (normalised, Notebook 01), at the price of two extra joins in every query that groups by category.

In [ ]:
for (t,) in dw.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
    links = [f"{fk[3]} -> {fk[2]}" for fk in dw.execute(f"PRAGMA foreign_key_list({t})")]
    print(f"{t:<16}", ", ".join(dict.fromkeys(links)) or "(no foreign keys)")

That listing is the schema diagram in words: `fact_sales` in the centre, pointing to six dimensions (three times to `dim_date`), with the product chain `dim_product -> dim_subcategory -> dim_category` and the customer chain `dim_customer -> dim_geography` forming the snowflake's arms. Exercise 3 flattens the product chain into a star and measures the difference.

## 5. A view for the analysts

Every analytical query of this warehouse starts with the same joins. A **view** stores a query under a name, so that analysts can treat it as one flat table and never write the joins again. Nothing is copied: the joins run each time the view is read.

In [ ]:
dw.executescript('''
DROP VIEW IF EXISTS v_sales;
CREATE VIEW v_sales AS
SELECT f.order_number, f.revenue, f.cost, f.quantity,
       d.full_date, d.year, d.quarter, d.month,
       p.product, s.subcategory, c.category,
       cu.gender, cu.yearly_income, cu.education, g.country
FROM fact_sales AS f
JOIN dim_date AS d        ON d.date_key = f.order_date_key
JOIN dim_product AS p     ON p.product_key = f.product_key
JOIN dim_subcategory AS s ON s.subcategory_key = p.subcategory_key
JOIN dim_category AS c    ON c.category_key = s.category_key
JOIN dim_customer AS cu   ON cu.customer_key = f.customer_key
JOIN dim_geography AS g   ON g.geography_key = cu.geography_key;
''')

def show(sql, **kw):
    return pd.read_sql_query(sql, dw, **kw)

show("SELECT * FROM v_sales LIMIT 3")

Now the moves of Notebook 04 on real data. **Roll up**: revenue by year.

In [ ]:
show("SELECT year, COUNT(*) AS lines, ROUND(SUM(revenue)) AS revenue FROM v_sales GROUP BY year ORDER BY year")

**Drill down**: from years to the quarters of 2013, by category.

In [ ]:
show('''SELECT quarter, category, ROUND(SUM(revenue)) AS revenue
        FROM v_sales WHERE year = 2013
        GROUP BY quarter, category ORDER BY quarter, revenue DESC''')

**Slice**: one country only, and a measure that needs two columns: the gross margin, revenue minus cost, as a share of revenue.

In [ ]:
show('''SELECT category, ROUND(SUM(revenue)) AS revenue,
               ROUND(100.0 * (SUM(revenue) - SUM(cost)) / SUM(revenue), 1) AS margin_pct
        FROM v_sales WHERE country = 'Germany'
        GROUP BY category ORDER BY revenue DESC''')

Bicycles bring almost all of the revenue. Accessories earn a clearly higher margin on each sale, and clothing a lower one. A sentence like that is the purpose of the whole warehouse.

## 6. How fast, and why?

### Intuition first
Notebook 03 showed that a database answers *"find these few rows"* by scanning everything unless an index lets it jump. The warehouse has one index already, created automatically for each primary key. Let us look at a typical lookup: all the sales of one customer.

In [ ]:
one_customer = "SELECT SUM(revenue) FROM fact_sales WHERE customer_key = 11000"

def plan(sql):
    return [row[-1] for row in dw.execute("EXPLAIN QUERY PLAN " + sql)]

def timed(sql, repeats=50):
    started = time.perf_counter()
    for i in range(repeats):
        dw.execute(sql).fetchall()
    return (time.perf_counter() - started) / repeats * 1000

print(plan(one_customer), f"{timed(one_customer):.2f} ms")
dw.execute("CREATE INDEX ix_sales_customer ON fact_sales(customer_key)")
print(plan(one_customer), f"{timed(one_customer):.2f} ms")

`SCAN` reads all sixty thousand rows; `SEARCH ... USING INDEX` goes straight to the handful that belong to the customer. But a warehouse is mostly asked the *other* kind of question, the one that reads everything:

In [ ]:
by_category = "SELECT category, ROUND(SUM(revenue), 2) FROM v_sales GROUP BY category"
print(f"{timed(by_category, 5):.0f} ms per run of the full analytical query")

No index helps a query that needs every row. What does help is to compute the answer once and keep it: a **summary table** (or *aggregate table*). Reports that are opened every morning are typically served this way.

In [ ]:
dw.executescript('''
DROP TABLE IF EXISTS sum_month_category;
CREATE TABLE sum_month_category AS
SELECT year, month, category, SUM(revenue) AS revenue, SUM(cost) AS cost, COUNT(*) AS lines
FROM v_sales GROUP BY year, month, category;
''')
from_summary = "SELECT category, ROUND(SUM(revenue), 2) FROM sum_month_category GROUP BY category"
print("rows in the summary:", dw.execute("SELECT COUNT(*) FROM sum_month_category").fetchone()[0])
print(f"from the facts:  {timed(by_category, 5):.1f} ms")
print(f"from the summary: {timed(from_summary):.2f} ms")
print("same answer:", sorted(dw.execute(by_category).fetchall()) == sorted(dw.execute(from_summary).fetchall()))

The summary gives the same answer from a few dozen rows instead of sixty thousand joined rows, and so it is several hundred times faster. The price is that it must be rebuilt whenever new sales arrive, and it can only answer the questions it was built for. Cloud warehouses offer the same trade under the name **materialised view**.

## 7. Where we stand

| The need | The tool |
|---|---|
| The grain must stay true | a primary key on the fact table |
| Every key must find its partner | foreign keys, enforced, and loading from the outside in |
| One date table for several dates of an event | a role-playing dimension |
| Category names stored once | a snowflaked dimension, at the cost of extra joins |
| Analysts should not write joins | a view |
| Fast lookups of a few rows | an index |
| Fast answers to the same big question | a summary table (a materialised view) |

The warehouse is relational to the core: fixed tables, fixed columns, joins. The next notebook asks what the same data looks like when it is stored the way a document database would store it, and what that gains and loses.

---
## ✏️ Exercises

### Exercise 1 (Three business questions)
Using `v_sales`, answer: (a) the five best-selling **products** by revenue in 2013; (b) the average revenue **per order** in each country; (c) the share of revenue from each **education** level.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
display(show('''SELECT product, ROUND(SUM(revenue)) AS revenue FROM v_sales WHERE year = 2013
                GROUP BY product ORDER BY revenue DESC LIMIT 5'''))
display(show('''SELECT country, ROUND(SUM(revenue) / COUNT(DISTINCT order_number), 1) AS revenue_per_order
                FROM v_sales GROUP BY country ORDER BY revenue_per_order DESC'''))
show('''SELECT education, ROUND(100.0 * SUM(revenue) / (SELECT SUM(revenue) FROM v_sales), 1) AS share_pct
        FROM v_sales GROUP BY education ORDER BY share_pct DESC''')
```

*In (b) the denominator is the number of distinct orders, not of rows: the grain of the fact table is the order line, so counting rows would give revenue per line.*

</details>

### Exercise 2 (Role-playing dates)
How many days does an order take to ship? Join `fact_sales` to `dim_date` **twice**, once as the order date and once as the ship date, and show the average number of days between them per year. (Hint: `julianday(a) - julianday(b)` gives days in SQLite.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
show('''SELECT od.year,
               ROUND(AVG(julianday(sd.full_date) - julianday(od.full_date)), 1) AS days_to_ship
        FROM fact_sales AS f
        JOIN dim_date AS od ON od.date_key = f.order_date_key
        JOIN dim_date AS sd ON sd.date_key = f.ship_date_key
        GROUP BY od.year ORDER BY od.year''')
```

*The same table joined under two aliases plays two roles. The answer is the same every year, which itself is a finding: in this sample data shipping always takes the same time.*

</details>

### Exercise 3 (From snowflake to star)
Create `dim_product_flat` with one row per product and the columns `product_key`, `product`, `subcategory` and `category` (a pure star dimension). Then time *revenue by category* through the snowflake and through the flat dimension.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
dw.executescript('''
DROP TABLE IF EXISTS dim_product_flat;
CREATE TABLE dim_product_flat AS
SELECT p.product_key, p.product, s.subcategory, c.category
FROM dim_product AS p
LEFT JOIN dim_subcategory AS s ON s.subcategory_key = p.subcategory_key
LEFT JOIN dim_category AS c    ON c.category_key = s.category_key;
''')
snowflake = '''SELECT c.category, ROUND(SUM(f.revenue), 2) FROM fact_sales AS f
               JOIN dim_product AS p ON p.product_key = f.product_key
               JOIN dim_subcategory AS s ON s.subcategory_key = p.subcategory_key
               JOIN dim_category AS c ON c.category_key = s.category_key GROUP BY c.category'''
star = '''SELECT p.category, ROUND(SUM(f.revenue), 2) FROM fact_sales AS f
          JOIN dim_product_flat AS p ON p.product_key = f.product_key GROUP BY p.category'''
print(f"snowflake: {timed(snowflake, 5):.1f} ms   star: {timed(star, 5):.1f} ms")
print("same answer:", sorted(dw.execute(snowflake).fetchall()) == sorted(dw.execute(star).fetchall()))
```

*The difference is small: the two extra joins are against tables of 37 and 4 rows, which cost almost nothing next to reading sixty thousand facts. The star's real advantage here is simplicity, one join instead of three for every analyst. With large dimensions, or on engines where joins are expensive, the speed difference grows. As always, measure before redesigning.*

</details>

### Exercise 4 (A summary that stays correct, a little harder)
New sales arrive for March 2014. Insert one order line into `fact_sales` (any existing customer and product, order date 20140310, ship 20140317, due 20140322). Show that `sum_month_category` is now **stale**, then rebuild only the affected month: delete its rows from the summary and insert them again from `v_sales`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
dw.execute('''INSERT INTO fact_sales VALUES ('SO99990', 1, 310, 11000, 6, 1, 20140310, 20140317, 20140322,
                                             1, 3578.27, 2171.29, 3578.27, 286.26, 89.46)''')
fresh = "SELECT ROUND(SUM(revenue), 2) FROM v_sales WHERE year = 2014 AND month = 3"
stale = "SELECT ROUND(SUM(revenue), 2) FROM sum_month_category WHERE year = 2014 AND month = 3"
print("facts:", dw.execute(fresh).fetchone()[0], "  summary:", dw.execute(stale).fetchone()[0])

dw.executescript('''
DELETE FROM sum_month_category WHERE year = 2014 AND month = 3;
INSERT INTO sum_month_category
SELECT year, month, category, SUM(revenue), SUM(cost), COUNT(*)
FROM v_sales WHERE year = 2014 AND month = 3 GROUP BY year, month, category;
''')
print("facts:", dw.execute(fresh).fetchone()[0], "  summary:", dw.execute(stale).fetchone()[0])
```

*Rebuilding only the month that changed is the incremental, idempotent refresh of Notebooks 10 and 12: run it twice and the summary is still right.*

</details>

## 🔑 Key takeaways

- Write the `CREATE TABLE` statements yourself: **primary keys** fix the grain and **foreign keys** keep every link valid.
- Load **from the outside in**: dimensions first, the fact table last.
- Checking quality finds bad rows; **enforcing** it keeps them out. Many cloud warehouses declare keys without enforcing them.
- A **role-playing** dimension serves several dates of one event; a **snowflaked** dimension saves storage and costs joins.
- A **view** hides the joins; an **index** speeds up lookups; a **summary table** speeds up the big questions that are asked again and again, and must be refreshed.

**Next:** *Notebook 15: From Warehouse to Documents*, where the same sales are reshaped into the documents of Notebook 06, and we measure what the new shape makes easy and what it makes hard.

In [ ]:
dw.close()

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*